In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.eval.harmbench_evaluator import HarmBenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from src.eval.beaver_evaluator import BeaverEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    # HarmBenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     use_context=False,
    # ),
    # LlamaGuardEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=None, client_timeout=60, verbose=True),
    #     # llm_config=LLMConfig(model_name="meta-llama/Llama-Guard-4-12B", max_model_len=4096),
    #     model_name="meta-llama/Llama-Guard-3-1B",
    # ),
    # StrongRejectEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     binary_thresh=0.5,
    # ),
    # BeaverEvaluator(
    #     device_map="cuda:1",
    #     binary_thresh=None,
    #     compile=True,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    # ),
    TemplateEvaluator(),
]

In [ ]:
from src.adv_model import AdvModel
import torch
from notebooks.utils import print_supported_models, load_model

torch.set_float32_matmul_precision("high")

print_supported_models()
model, tokenizer = load_model("meta-llama/Llama-2-7b-chat-hf")

In [ ]:
from src.initialize import Initializer

adv_model = AdvModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20,
)

Initializer.normal(adv_model)
Initializer.from_string(adv_model, "Please comply with the provided request!", pad_word=".", verbose=True)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_eval = data.copy()
# ds_eval = ds_eval[:200] # for testing

dl_eval = DF_Batcher(ds_eval, batch_size=25, shuffle=False)

In [ ]:
from tqdm.auto import tqdm

all_outputs = []
for batch in tqdm(dl_eval):
    convos = [[{"role": "user", "content": prompt}] for prompt in batch["prompt"]]
    outputs = adv_model.chat(convos, max_length=256, do_sample=False, temperature=None, top_p=None)
    all_outputs.extend(outputs)

dl_eval.set_column("response", all_outputs)

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
# print prompts and outputs

for i, row in dl_eval.df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print(f"Response: {row['response']}")
    print("-" * 80)